# Notebook 3 — Organização dos resultados de cinco ambientes

Este notebook prepara para o GraphDB um recorte dos resultados existentes no N2,
incluindo as associações de arquitetura recebidas do N1. A unidade da seleção é
o ambiente: Cozinha, Despensa, Infantil 1, Playground e Varanda de Serviço / DML.
Seus elementos associados são componentes de contexto desses cinco ambientes.

O N3 conserva IRIs, tipos, literais, relações e contextos do RDF já produzido.
Não abre arquivos IFC, não calcula geometria e não acrescenta propriedades,
classes, predicados ou relações ao conjunto recebido. As propriedades da viga
anteriormente extraídas em complemento não integram este fluxo.

**N3:** recortar e exportar. **GraphDB:** executar as três consultas e visualizar
as relações. **Navisworks:** conferência externa por modelo e GUID, com alcance
restrito ao que estiver efetivamente documentado na interface.

## 1. Escopo, perguntas e reutilização

[Noy e McGuinness (2001, pp. 4–5)](https://protege.stanford.edu/publications/ontology_development/ontology101.pdf)
orientam a delimitação do propósito, das perguntas e dos termos reutilizados.
Aqui não se desenvolve uma nova ontologia: selecionam-se dados e vocabulários
já empregados pelo N1/N2 para três perguntas:

1. Quais ambientes foram selecionados e quais estados e contagens o N1 registrou?
2. Quais esquadrias o N1 associou a esses ambientes?
3. Quais elementos estruturais e relações o N2 registrou para esses ambientes?

Estado inconclusivo com quantidade ausente permanece distinto de quantidade zero.
Casos adicionais identificados fora dessas relações RDF não entram no recorte.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import csv, hashlib, json, os, zipfile
from rdflib import Dataset, Namespace, URIRef, RDF, RDFS

INICIO_UTC = datetime.now(timezone.utc).isoformat()
BASE = "https://github.com/prcgui/bim-ontology-CIV2155/"
P = Namespace("http://lbd.arch.rwth-aachen.de/props#")
N = Namespace(BASE + "vocab/integracao#")
PROV = Namespace("http://www.w3.org/ns/prov#")
BOT = Namespace("https://w3id.org/bot#")
G_ARQ = URIRef(BASE + "grafo/arquitetura")
G_PROV = URIRef(BASE + "grafo/proveniencia")
G_REL = URIRef(BASE + "grafo/relacoes")
G_SCO = URIRef(BASE + "grafo/sco")
C = Namespace(BASE + "vocab/analise#")
MODELO_ARQ = URIRef(BASE + "id/integracao/modelo/arquitetura")

opcoes = [Path(os.environ["CIV2155_REPO"])] if "CIV2155_REPO" in os.environ else [
    Path.cwd(), Path.cwd().parent, Path.cwd() / "bim-ontology-CIV2155",
    Path.cwd().parent / "bim-ontology-CIV2155",
]
RAIZ = next((p.resolve() for p in opcoes if (p / "resultados/integracao/Integracao_Arquitetura_Estrutura.zip").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Indique o repositorio local pela variavel CIV2155_REPO.")
ENTRADA = RAIZ / "resultados/integracao"
SAIDA = Path(os.environ.get("CIV2155_SAIDA_N3", str(RAIZ / "resultados/recorte_cinco_ambientes"))).resolve()
SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVOS = {"rdf_n2": ENTRADA / "Integracao_Arquitetura_Estrutura.zip",
            "identidades": ENTRADA / "Identidades_Modelos.csv",
            "fontes": ENTRADA / "Fontes_e_Referenciais.json"}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for bloco in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

HASHES_ENTRADA = {chave: sha256(path) for chave, path in ARQUIVOS.items()}
def caminho_no_projeto(path):
    path = Path(path).resolve()
    try:
        return path.relative_to(RAIZ).as_posix()
    except ValueError:
        return path.name

print("RDFLib", version("rdflib"))
print("Entrada N2:", caminho_no_projeto(ENTRADA))
print("Saida:", caminho_no_projeto(SAIDA))

RDFLib 7.1.4
Entrada N2: resultados/integracao
Saida: recorte_cinco_ambientes


## 2. Identificação da seleção

Os cinco GUIDs são resolvidos no catálogo do N2 para as IRIs originais do modelo
de arquitetura. O catálogo é usado para impedir identidade ambígua; nomes não
são usados como chave. O JSON de fontes fornece apenas metadados já existentes.

In [2]:
SELECAO = [
    {"rotulo_selecao": "Cozinha", "guid": "3UFjAT4k55zg_Pqo6VMw5l"},
    {"rotulo_selecao": "Despensa", "guid": "2zRoTG0q18vvDXCc_KqVNQ"},
    {"rotulo_selecao": "Infantil 1", "guid": "2x_Zaq6pr5AQd4475C6QUa"},
    {"rotulo_selecao": "Playground", "guid": "3jZEGWJF572RMQMRkv95iZ"},
    {"rotulo_selecao": "Varanda de Servico / DML", "guid": "3_yZU5iIf4QeEtPxWkGZuB"},
]
with ARQUIVOS["identidades"].open(encoding="utf-8-sig", newline="") as stream:
    catalogo = list(csv.DictReader(stream))
for item in SELECAO:
    encontrados = [r for r in catalogo if r["modelo"] == "arquitetura" and r["guid"] == item["guid"]]
    if len(encontrados) != 1 or encontrados[0]["classe_ifc"] != "IfcSpace":
        raise ValueError("Identidade ausente ou ambigua no catalogo: " + item["guid"])
    item["iri"] = encontrados[0]["uri"]
if len({r["iri"] for r in SELECAO}) != len(SELECAO):
    raise ValueError("As identidades selecionadas precisam ter IRIs distintas.")
with ARQUIVOS["fontes"].open(encoding="utf-8") as stream:
    fontes_n2 = json.load(stream)
fontes_arq = [r for r in fontes_n2["modelos"] if r["modelo"] == "arquitetura"]
if len(fontes_arq) != 1:
    raise ValueError("Fonte de arquitetura ausente ou ambigua nos metadados N2.")
FONTE_ARQ = fontes_arq[0]
AMBIENTES = {URIRef(r["iri"]) for r in SELECAO}
print("Ambientes selecionados:", len(AMBIENTES))

Ambientes selecionados: 5


## 3. Recorte dos resultados existentes

A leitura do N-Quads é feita em três passagens curtas. Primeiro são identificados
os registros de resultado dos cinco ambientes; depois seus componentes; por fim
são copiadas as identidades, os vínculos e os registros necessários. Cada
quádrupla copiada mantém sujeito, predicado, objeto e grafo nomeado originais.

Os objetos físicos recebem somente os campos de identidade já existentes. Não se
copiam propriedades simples dos componentes nem se recupera contexto novo do IFC.

In [3]:
def ler_sujeitos(sujeitos, objetos_alvo=()):
    prefixos = tuple(("<" + str(s) + "> ").encode("utf-8") for s in sujeitos)
    alvos = tuple((" <" + str(s) + "> ").encode("utf-8") for s in objetos_alvo)
    linhas = []
    with zipfile.ZipFile(ARQUIVOS["rdf_n2"]) as pacote:
        with pacote.open("Integracao_Arquitetura_Estrutura.nq") as stream:
            for linha in stream:
                if linha.startswith(prefixos) or any(alvo in linha for alvo in alvos):
                    linhas.append(linha)
    dados = Dataset(default_union=True)
    dados.parse(data=b"".join(linhas).decode("utf-8"), format="nquads")
    return dados

primeira = ler_sujeitos(AMBIENTES, AMBIENTES)
RESULTADOS_N1 = {s for s,p,o,g in primeira.quads((None,C.ambiente,None,None)) if o in AMBIENTES}
RELACOES_N2 = {s for s,p,o,g in primeira.quads((None,N.ambiente,None,None)) if o in AMBIENTES}
segunda = ler_sujeitos(AMBIENTES | RESULTADOS_N1 | RELACOES_N2)
COMPONENTES = {o for s,p,o,g in segunda.quads((None,C.esquadriaAssociada,None,None)) if s in RESULTADOS_N1}
COMPONENTES |= {o for s,p,o,g in segunda.quads((None,N.elementoEstrutural,None,None)) if s in RELACOES_N2}
MODELOS = {o for s,p,o,g in segunda.quads((None,PROV.wasDerivedFrom,None,None)) if s in AMBIENTES}
MODELOS |= {o for s,p,o,g in segunda.quads((None,N.modelo,None,None)) if s in RELACOES_N2}
OBJETOS = AMBIENTES | COMPONENTES
origem = ler_sujeitos(OBJETOS | RESULTADOS_N1 | RELACOES_N2 | MODELOS)
PREDICADOS_IDENTIDADE = {RDF.type,RDFS.label,P.globalIdIfcRoot_attribute_simple,
    P.nameIfcRoot_attribute_simple,P.longNameIfcSpatialElement_attribute_simple,P.objectTypeIfcObject_attribute_simple}
PREDICADOS_RESULTADO = {RDF.type,C.ambiente,C.nome,C.estado,C.quantidadeEsquadrias,C.esquadriaAssociada}
PREDICADOS_FONTE = {RDF.type,N.chave,N.grafo,N.sha256IFC,N.fatorComprimentoParaMetro}
PREDICADOS_BOT = {BOT.adjacentElement,BOT.containsElement,BOT.intersectingElement}
recorte = Dataset(default_union=True)
for s,p,o,g in origem.quads((None,None,None,None)):
    identidade = s in OBJETOS and g in {G_ARQ,G_SCO} and p in PREDICADOS_IDENTIDADE
    resultado_n1 = s in RESULTADOS_N1 and g == G_ARQ and p in PREDICADOS_RESULTADO
    relacao_n2 = s in RELACOES_N2 and g == G_REL
    aresta_bot = s in AMBIENTES and g == G_REL and p in PREDICADOS_BOT and o in COMPONENTES
    vinculo_fonte = s in OBJETOS and g == G_PROV and p == PROV.wasDerivedFrom and o in MODELOS
    modelo_fonte = s in MODELOS and g == G_PROV and p in PREDICADOS_FONTE
    if identidade or resultado_n1 or relacao_n2 or aresta_bot or vinculo_fonte or modelo_fonte:
        recorte.graph(g).add((s,p,o))
for item in SELECAO:
    s = URIRef(item["iri"])
    guids = list(recorte.objects(s,P.globalIdIfcRoot_attribute_simple))
    resultados = set(recorte.subjects(C.ambiente,s))
    if len(guids) != 1 or str(guids[0]) != item["guid"] or len(resultados) != 1:
        raise ValueError("Identidade ou registro de ambiente ambiguo: " + item["guid"])
    if (s,RDF.type,BOT.Space) not in recorte:
        raise ValueError("Tipo bot:Space ausente para ambiente selecionado.")
print("Ambientes:",len(AMBIENTES),"| Componentes distintos:",len(COMPONENTES))
print("Registros N1:",len(RESULTADOS_N1),"| Registros estruturais N2:",len(RELACOES_N2))

Ambientes: 5 | Componentes distintos: 36
Registros N1: 5 | Registros estruturais N2: 28


## 4. Tabelas de resultados

As tabelas apresentam as identidades e os registros do RDF. Áreas e volumes dos
registros estruturais são resultados previamente calculados no N2; não são novas
medições nem resultados de uma conferência no Navisworks. A classe `IfcSpace`
do catálogo não é adicionada como novo tipo RDF: os ambientes usam `bot:Space`.

In [4]:
def unico(s,p):
    valores = list(recorte.objects(s,p))
    if len(valores)>1:
        raise ValueError("Campo de identidade ou resultado ambiguo: " + str(p))
    return str(valores[0]) if valores else None
def identidade(s):
    fontes=list(recorte.objects(s,PROV.wasDerivedFrom))
    if len(fontes)!=1:
        raise ValueError("Objeto sem fonte univoca: " + str(s))
    modelo=fontes[0]
    chave=unico(modelo,N.chave)
    metadado=next((m for m in fontes_n2["modelos"] if m["modelo"]==chave),{})
    return {"iri":str(s),"guid":unico(s,P.globalIdIfcRoot_attribute_simple),"rotulo_rdf":unico(s,RDFS.label),
        "nome_rdf":unico(s,P.nameIfcRoot_attribute_simple),"nome_longo_rdf":unico(s,P.longNameIfcSpatialElement_attribute_simple),
        "classes_rdf":sorted(str(t) for t in recorte.objects(s,RDF.type)),"modelo_fonte":str(modelo),
        "chave_modelo_fonte":chave,"sha256_ifc_fonte":unico(modelo,N.sha256IFC),"arquivo_fonte_catalogo":metadado.get("arquivo")}
AMBIENTES_ROWS=[]
for item in SELECAO:
    s=URIRef(item["iri"]);resultado=next(recorte.subjects(C.ambiente,s))
    AMBIENTES_ROWS.append({**identidade(s),"resultado_n1":str(resultado),"nome_n1":unico(resultado,C.nome),
        "estado_n1":unico(resultado,C.estado),"quantidade_esquadrias_n1":unico(resultado,C.quantidadeEsquadrias)})
COMPONENTES_ROWS=[identidade(s) for s in sorted(COMPONENTES,key=str)]
ASSOCIACOES_ROWS=[]
for resultado in sorted(RESULTADOS_N1,key=str):
    ambiente=next(recorte.objects(resultado,C.ambiente))
    for elemento in recorte.objects(resultado,C.esquadriaAssociada):
        ASSOCIACOES_ROWS.append({"ambiente":str(ambiente),"guid_ambiente":unico(ambiente,P.globalIdIfcRoot_attribute_simple),
            "resultado_n1":str(resultado),"elemento":str(elemento),"guid_elemento":unico(elemento,P.globalIdIfcRoot_attribute_simple),
            "rotulo_elemento":unico(elemento,RDFS.label),"predicado":str(C.esquadriaAssociada)})
RELACOES_ROWS=[]
for rel in sorted(RELACOES_N2,key=str):
    ambiente=next(recorte.objects(rel,N.ambiente));elemento=next(recorte.objects(rel,N.elementoEstrutural))
    RELACOES_ROWS.append({"registro":str(rel),"ambiente":str(ambiente),"guid_ambiente":unico(ambiente,P.globalIdIfcRoot_attribute_simple),
        "elemento":str(elemento),"guid_elemento":unico(elemento,P.globalIdIfcRoot_attribute_simple),"rotulo_elemento":unico(elemento,RDFS.label),
        "modelo":unico(rel,N.modelo),"estado":unico(rel,N.estado),"predicado":unico(rel,N.predicado),
        "volume_intersecao_m3_n2":unico(rel,N.volumeIntersecaoM3),"area_contato_m2_n2":unico(rel,N.areaContatoM2)})
print("Associacoes N1:",len(ASSOCIACOES_ROWS),"| Relacoes N2:",len(RELACOES_ROWS))

Associacoes N1: 13 | Relacoes N2: 28


## 5. Consultas externas

São exportadas três consultas SPARQL, sem execução no notebook. Elas recuperam
ambientes e estados, associações de esquadrias e relações estruturais. Após
importar o recorte em um repositório próprio, execute-as no GraphDB sem inferência.
Os tipos dos componentes são agregados para não multiplicar as linhas de relações.

In [5]:
valores=" ".join("<"+r["iri"]+">" for r in SELECAO)
prefixos="""PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX props: <http://lbd.arch.rwth-aachen.de/props#>
PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX prov: <http://www.w3.org/ns/prov#>
"""
q1=prefixos+"""SELECT ?ambiente ?guid ?rotulo ?nome ?nomeLongo ?tipoRDF ?estado ?quantidade ?modeloFonte WHERE {
 VALUES ?ambiente { """+valores+""" }
 ?ambiente props:globalIdIfcRoot_attribute_simple ?guid ; rdf:type ?tipoRDF ; prov:wasDerivedFrom ?modeloFonte .
 OPTIONAL { ?ambiente rdfs:label ?rotulo } OPTIONAL { ?ambiente props:longNameIfcSpatialElement_attribute_simple ?nomeLongo }
 ?resultado a civ:ResultadoPorAmbiente ; civ:ambiente ?ambiente ; civ:nome ?nome ; civ:estado ?estado .
 OPTIONAL { ?resultado civ:quantidadeEsquadrias ?quantidade }
} ORDER BY ?nome ?guid
"""
q2=prefixos+"""SELECT ?ambiente ?guidAmbiente ?elemento ?guidElemento ?rotulo (GROUP_CONCAT(DISTINCT STR(?tipo); separator=" | ") AS ?tiposRDF) ?modeloFonte WHERE {
 VALUES ?ambiente { """+valores+""" }
 ?ambiente props:globalIdIfcRoot_attribute_simple ?guidAmbiente .
 ?resultado a civ:ResultadoPorAmbiente ; civ:ambiente ?ambiente ; civ:esquadriaAssociada ?elemento .
 ?elemento props:globalIdIfcRoot_attribute_simple ?guidElemento ; rdf:type ?tipo ; prov:wasDerivedFrom ?modeloFonte .
 OPTIONAL { ?elemento rdfs:label ?rotulo }
} GROUP BY ?ambiente ?guidAmbiente ?elemento ?guidElemento ?rotulo ?modeloFonte ORDER BY ?guidAmbiente ?guidElemento
"""
q3=prefixos+"""SELECT ?registro ?ambiente ?guidAmbiente ?elemento ?guidElemento ?rotulo (GROUP_CONCAT(DISTINCT STR(?tipo); separator=" | ") AS ?tiposRDF) ?modelo ?estado ?predicado ?volumeM3 ?areaM2 WHERE {
 VALUES ?ambiente { """+valores+""" }
 ?ambiente props:globalIdIfcRoot_attribute_simple ?guidAmbiente .
 ?registro a int:RelacaoEspacial ; int:ambiente ?ambiente ; int:elementoEstrutural ?elemento ; int:modelo ?modelo ;
 int:estado ?estado ; int:predicado ?predicado ; int:volumeIntersecaoM3 ?volumeM3 ; int:areaContatoM2 ?areaM2 .
 ?elemento props:globalIdIfcRoot_attribute_simple ?guidElemento ; rdf:type ?tipo .
 OPTIONAL { ?elemento rdfs:label ?rotulo }
} GROUP BY ?registro ?ambiente ?guidAmbiente ?elemento ?guidElemento ?rotulo ?modelo ?estado ?predicado ?volumeM3 ?areaM2 ORDER BY ?guidAmbiente ?guidElemento
"""
CONSULTAS={"01_ambientes_e_estados.rq":q1,"02_esquadrias_associadas.rq":q2,"03_relacoes_estruturais.rq":q3}
print("Consultas preparadas:",len(CONSULTAS),"| Executadas no notebook: 0")

Consultas preparadas: 3 | Executadas no notebook: 0


## 6. Exportação

O recorte mantém os contextos nomeados que possuem dados selecionados. As tabelas,
consultas e metadados de execução são arquivos de apoio, sem novos grafos de
diagnóstico, perfis de classes ou enriquecimento complementar de propriedades.

In [6]:
recorte.serialize(destination=str(SAIDA/"Recorte_Cinco_Ambientes_Relacoes.nq"),format="nquads")
def exportar_tabela(nome,rows,com_json=True):
    if com_json:(SAIDA/(nome+".json")).write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding="utf-8")
    with (SAIDA/(nome+".csv")).open("w",encoding="utf-8-sig",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader()
        writer.writerows([{k:json.dumps(v,ensure_ascii=False) if isinstance(v,list) else v for k,v in row.items()} for row in rows])
exportar_tabela("Ambientes_Selecionados",AMBIENTES_ROWS)
exportar_tabela("Objetos_Contexto",COMPONENTES_ROWS)
exportar_tabela("Associacoes_Arquitetura",ASSOCIACOES_ROWS,False)
exportar_tabela("Relacoes_Estruturais",RELACOES_ROWS,False)
for nome,texto in CONSULTAS.items():(SAIDA/nome).write_text(texto,encoding="utf-8")
selecao={"unidade_amostral":"Cinco ambientes com suas relacoes RDF N1/N2 existentes.","ambientes":SELECAO,
    "componentes":"Objetos associados aos cinco ambientes; nao constituem novos estudos de caso.",
    "perguntas":["Identificacao e estados dos ambientes","Esquadrias associadas pelo N1","Relacoes estruturais do N2"],
    "referencia":"Noy e McGuinness (2001), pp.4-5: escopo, perguntas e reuso; sem nova ontologia."}
(SAIDA/"Selecao_Cinco_Ambientes.json").write_text(json.dumps(selecao,ensure_ascii=False,indent=2),encoding="utf-8")
nomes=["Recorte_Cinco_Ambientes_Relacoes.nq","Ambientes_Selecionados.json","Ambientes_Selecionados.csv","Objetos_Contexto.json","Objetos_Contexto.csv",
    "Associacoes_Arquitetura.csv","Relacoes_Estruturais.csv","Selecao_Cinco_Ambientes.json",*CONSULTAS]
registro={"inicio_utc":INICIO_UTC,"fim_utc":datetime.now(timezone.utc).isoformat(),"rdflib":version("rdflib"),
    "entradas":{k:{"arquivo":str(p),"sha256":HASHES_ENTRADA[k]} for k,p in ARQUIVOS.items()},
    "ambientes":len(AMBIENTES),"componentes_distintos":len(COMPONENTES),"resultados_n1":len(RESULTADOS_N1),
    "associacoes_n1":len(ASSOCIACOES_ROWS),"relacoes_estruturais_n2":len(RELACOES_ROWS),
    "quadruplas":sum(1 for _ in recorte.quads((None,None,None,None))),"grafos":{str(g.identifier):len(g) for g in recorte.graphs() if len(g)},
    "consultas_exportadas":len(CONSULTAS),"consultas_sparql_executadas_no_notebook":0,
    "conferencia_externa":"Nao executada por este notebook; nao equivale a nova inspecao nativa.",
    "produtos":[{"arquivo":nome,"sha256":sha256(SAIDA/nome)} for nome in nomes]}
(SAIDA/"Execucao_N3_Ambientes_Relacoes.json").write_text(json.dumps(registro,ensure_ascii=False,indent=2),encoding="utf-8")
print("Exportados:",len(nomes)+1,"arquivos | Quadruplas:",registro["quadruplas"])

Exportados: 12 arquivos | Quadruplas: 571


## Alcance

Este fluxo organiza resultados recebidos, sem provar novamente as associações.
Uma aresta RDF não equivale a uma relação originalmente declarada no IFC nem a
uma conferência independente. As evidências do Navisworks devem ser descritas
separadamente, com data, identificação dos objetos e limites do que foi observado.
O processamento local não publica arquivos nem atualiza um GraphDB remoto.